# Chicago Traffic Crash Analysis (2018–2025)
## Notebook 1: Data collection and cleaning

**Author:** Armon Saeedvafa
**Date:** November 2026

This is the first of two notebooks. It:

1. downloads reported crashes, person-level injury records, the street network, traffic volumes, mobility data, community-area boundaries and population estimates from the City of Chicago and the Illinois Department of Transportation (IDOT);
2. checks data quality and documents each cleaning decision;
3. builds a road-segment **exposure model** that estimates how many vehicle-miles are driven on every street block; and
4. saves the prepared datasets to `data/processed/prepared_data.pkl`.

**Run this notebook first.** [Notebook 2 (`02_analysis.ipynb`)](02_analysis.ipynb) loads the saved file and contains the executive summary, analysis, maps, findings and recommendations.

### Project folders

| Folder | Contents | Uploaded to GitHub? |
|---|---|---|
| `data/raw/` | Files downloaded by this notebook (reused on later runs) | No (too large; anyone can recreate them by running this notebook) |
| `data/processed/` | `prepared_data.pkl`, the hand-off to Notebook 2 | No |
| `images/`, `docs/maps/` | Charts and interactive maps written by Notebook 2 | Yes |

## 1. Setup and data sources

The notebook uses Chicago's Traffic Crashes dataset, the associated Traffic Crashes – People dataset, official community-area boundaries, and community-area population estimates. For exposure, it also uses:

| Exposure measure | Source | What it represents |
|---|---|---|
| Street network | City of Chicago street centerlines | Every street block, with road class, address range, length and intersection nodes |
| Traffic volume | IDOT Annual Average Daily Traffic (AADT) segments | Two-way daily traffic on counted roads |
| Traffic volume | City of Chicago camera-based Average Daily Traffic counts (Oct 2020 onward) | Directional daily traffic near ~800 traffic and speed cameras |
| Time-of-week travel pattern | Transportation Network Provider (ride-hail) trips, 2019 and 2024 samples | Share of vehicle travel by hour of the week and by community area |
| Pedestrian activity | CTA 'L' station entries and bus-stop boardings | People walking to and from transit |
| Cycling activity | Divvy bike-share trips (2018–2019) | Bike trips by station and hour |

The crash dataset is treated as crash-level data. The People dataset contains person-level records, so it must be aggregated to the crash level before being joined to crash-level data. Community boundaries support spatial assignment, population estimates support a per-resident comparison, and the exposure data support per-trip (per-vehicle-mile) comparisons.

### Reproducibility

The download cells save each calendar year's data, and each exposure dataset, as a local file. On later runs, existing files are reused instead of downloaded again. Everything is saved under `data/raw/`. To refresh the data from the City or IDOT APIs, remove the relevant saved files first (crash and People files in `data/raw/chicago_crash_data/` and `data/raw/chicago_people_data/`; exposure, boundary and population files in `data/raw/chicago_exposure_data/`).

The required packages are listed in `requirements.txt` (install them with `pip install -r requirements.txt`). The first full run downloads several large datasets, so it can take a while; later runs reuse the saved files.

In [1]:
# Setup: libraries, display options, study-period constants and project folders.

# --- Standard-library tools (built into Python) ---
import json                     # read JSON text (the format web APIs return)
import os                       # walk through folders when searching for the project
import threading                # give each parallel download worker its own connection
import time                     # pause between retries (time.sleep) and time long downloads
from concurrent.futures import ThreadPoolExecutor, as_completed  # run several downloads at once
from pathlib import Path        # build file paths that work on Mac, Windows and Linux

# --- Third-party libraries (installed from requirements.txt) ---
import geopandas as gpd         # tables with a map-geometry column (points, lines, polygons)
import numpy as np              # fast maths on whole columns of numbers
import pandas as pd             # tables (DataFrames): load, clean, group and join data
import requests                 # download data from web APIs
from IPython.display import display              # show a table as a formatted table in the notebook
from shapely.geometry import LineString          # build a line from two points (camera count segments)

# Show more columns and a wider layout when a table is printed, so nothing important is hidden.
pd.set_option("display.max_columns", 100)       # show up to 100 columns before hiding any with "..."
pd.set_option("display.width", 140)             # allow printed tables up to 140 characters wide

# Study period, set once here so every later cell uses the same years.
START_YEAR = 2018                               # first calendar year analysed
END_YEAR = 2025                                 # last calendar year analysed
N_YEARS = END_YEAR - START_YEAR + 1             # number of years (8), used for annual averages


PROJECT_NAME = "chicago-traffic-crash-analysis"  # the name of the project's main folder


# Helper: check whether a folder is the project's main folder.
def is_project_folder(folder):
    """True if `folder` is the project's main folder (it holds requirements.txt and a notebooks/ folder)."""
    # Both files must exist: requirements.txt (a file) and notebooks (a folder).
    return (folder / "requirements.txt").is_file() and (folder / "notebooks").is_dir()


# Helper: find the project's main folder, wherever Jupyter or VS Code was started from.
def find_project_root():
    """Return the project's main folder, so data, images and maps are always saved inside the project.

    1. The folder the notebook is running in, or any folder above it (the usual case: running from notebooks/).
    2. Otherwise the usual places for the project in your home folder (GitHub Desktop saves to Documents/GitHub).
    3. Otherwise a search of your home folder, up to 4 levels deep, skipping hidden and system folders."""
    here = Path.cwd().resolve()                                     # the folder the notebook is running in
    for folder in [here, *here.parents]:                            # this folder, then each folder above it
        if is_project_folder(folder):                               # is this the project folder?
            return folder                                           # yes: stop and return it
    home = Path.home()                                              # your home folder, e.g. /Users/armon
    for place in ["Documents/GitHub", "GitHub", "Documents", "Desktop", "Downloads", ""]:   # common locations
        if is_project_folder(home / place / PROJECT_NAME):          # e.g. ~/Documents/GitHub/<project>
            return home / place / PROJECT_NAME                      # found it: return it
    matches = []                                                    # project folders found by the search
    for root, subfolders, _ in os.walk(home):                       # visit every folder under home in turn
        root = Path(root)                                           # turn the folder name into a Path
        depth = len(root.relative_to(home).parts)                   # how many levels below home we are
        # Edit the list in place to skip hidden folders, system/program folders, and anything too deep.
        subfolders[:] = [s for s in subfolders if depth < 4 and not s.startswith(".")
                         and s not in {"Library", "Applications", "anaconda3", "miniconda3", "node_modules"}]  # nor these folders
        if root.name == PROJECT_NAME and is_project_folder(root):   # right name and right contents?
            matches.append(root)                                    # remember it
            subfolders[:] = []                                      # no need to look inside it
    if matches:                                                     # at least one copy was found
        matches.sort(key=lambda folder: not (folder / ".git").exists())   # the copy tracked by Git first
        if len(matches) > 1:                                        # more than one copy: say which is used
            print("Several project copies found; using the one tracked by GitHub Desktop:",  # several copies: list them
                  *[f"  {m}" for m in matches], sep="\n")           # one folder per line
        return matches[0]                                           # the preferred copy
    raise FileNotFoundError(                                        # not found: stop with a clear instruction
        f"Could not find the {PROJECT_NAME} folder. Open this notebook from that folder's notebooks/ folder, "
        "or replace find_project_root() below with Path('/full/path/to/chicago-traffic-crash-analysis').")


# Project folders. `/` joins folder names into a path.
PROJECT_ROOT = find_project_root()                   # the project's main folder
RAW_DIR = PROJECT_ROOT / "data" / "raw"              # downloaded files (reused on later runs)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"  # prepared data handed to Notebook 2
RAW_DIR.mkdir(parents=True, exist_ok=True)           # create the folder if missing; do nothing if it exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)     # same for the processed-data folder
print(f"Project folder: {PROJECT_ROOT}")             # check this is your chicago-traffic-crash-analysis folder

# Safety check: if earlier runs saved the downloads outside the project (e.g. in your home folder) and the
# project's data/raw folder is still empty, stop now instead of downloading everything again.
OLD_RAW_DIR = Path.home() / "data" / "raw"           # where earlier runs saved downloads by mistake
# Stop if that old folder exists, is not the project's own folder, and the project's raw folder is empty.
if OLD_RAW_DIR.is_dir() and OLD_RAW_DIR.resolve() != RAW_DIR.resolve() and not any(RAW_DIR.iterdir()):
    raise FileExistsError(                           # stop the cell with instructions
        f"Saved downloads were found in {OLD_RAW_DIR.parent}, outside the project. Move that 'data' folder into "
        f"{PROJECT_ROOT} (replace the empty one there), then run this cell again.")

Project folder: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis


## 2. Collect and understand the data

### 2.1 Download or load the crash-level dataset

This retrieves selected fields for 2018–2025. Each year is saved separately so later runs can reuse completed downloads.

In [2]:
# Shared download helpers, then download (or load) the Traffic Crashes dataset, one calendar year at a time.

# The City of Chicago data portal (Socrata software). Each dataset is <SOCRATA_BASE>/<dataset ID>.json.
SOCRATA_BASE = "https://data.cityofchicago.org/resource"
HTTP = requests.Session()                                           # one reusable connection: faster than
HTTP.headers.update({"User-Agent": "ChicagoCrashCapstone/1.0"})     # reconnecting; the header names this program


# Helper: download one web address, trying again when the server is busy or the connection drops.
def request_with_retries(url, params=None, timeout=(20, 300), max_retries=8):
    """GET a URL, retrying on rate limits, server errors and dropped connections.
    Waits 5, 10, 20, 40, 80, then 120 seconds between attempts (about 6.5 minutes in total), because the City
    portal sometimes answers 503 "Service Unavailable" for several minutes when it is busy."""
    for attempt in range(max_retries):                              # attempt = 0, 1, 2, ... max_retries - 1
        wait_seconds = min(5 * (2 ** attempt), 120)                 # double the wait each time, at most 120 s
        last_attempt = attempt == max_retries - 1                   # True on the final try
        try:                                                        # try the download; errors jump to except
            # timeout=(20, 300): wait up to 20 s to connect and 300 s for the data to arrive.
            response = HTTP.get(url, params=params, timeout=timeout)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as error:  # no reply at all
            if last_attempt:                                        # no tries left: stop with a clear message
                raise RuntimeError(f"Could not reach the data portal after {max_retries} attempts. "  # stop with a clear message
                                   "Check your internet connection, then rerun this cell.") from error
            # Report the problem, wait, then go back to the top of the loop for the next attempt.
            print(f"Connection issue; retrying in {wait_seconds} seconds (attempt {attempt + 1} of {max_retries}).")
            time.sleep(wait_seconds)                                # pause before trying again
            continue                                                # skip the rest of the loop body
        if response.status_code == 400:  # 400 = the query itself is wrong, so retrying will not help
            print("Invalid API query:", response.text[:2000])       # show the server's explanation
            response.raise_for_status()                             # stop with an error
        if response.status_code == 429 or response.status_code >= 500:  # 429 = too many requests; 5xx = server
            if last_attempt:                                              # busy or down: wait and retry
                raise RuntimeError(                                 # no tries left: stop with instructions
                    f"The data portal is temporarily unavailable (HTTP {response.status_code}) after {max_retries} "
                    "attempts. Wait 15-30 minutes and rerun this cell. Years already saved are reused, so the "
                    "download continues from the year that failed.")
            print(f"Server busy (HTTP {response.status_code}); retrying in {wait_seconds} seconds "  # tell the user what is happening
                  f"(attempt {attempt + 1} of {max_retries}).")     # tell the user what is happening
            time.sleep(wait_seconds)                                # pause before trying again
            continue                                                # go to the next attempt
        response.raise_for_status()                                 # stop on any other error code
        return response                                             # success (status 200)


# Helper: download every row of a query, one page at a time.
def socrata_all(dataset_id, params, page_size=50000, show_progress=False):
    """Page through a City of Chicago (Socrata) query and return every row as a DataFrame."""
    pages, offset = [], 0                       # pages collects each batch; offset = rows already received
    while True:                                 # keep asking for pages until one comes back short or empty
        # Ask for the next page: $limit rows, skipping the first $offset rows. dict(params, **{...}) copies
        # the query and adds the paging settings without changing the original.
        batch = request_with_retries(f"{SOCRATA_BASE}/{dataset_id}.json",
                                     dict(params, **{"$limit": page_size, "$offset": offset})).json()  # then read the JSON reply
        if not batch:                           # an empty page means there is nothing left
            break                               # leave the loop
        pages.append(pd.DataFrame(batch))       # turn the list of records into a table and keep it
        offset += len(batch)                    # the next request starts after these rows
        if show_progress:                       # only print progress when asked to
            print(f"Retrieved {offset:,} records for this year.")  # :, adds thousands separators
        if len(batch) < page_size:              # a short page is the last page
            break                               # leave the loop
        time.sleep(0.5)                         # brief pause to be polite to the server
    # Stack all pages into one table (ignore_index renumbers the rows 0, 1, 2, ...).
    return pd.concat(pages, ignore_index=True) if pages else pd.DataFrame()


# Helper: load each year from its saved file, or download and save it the first time.
def load_years(dataset_id, columns, folder, prefix, label, page_size, dtypes):
    """Load each year's saved CSV, or download and save that year if it is not saved yet."""
    folder.mkdir(parents=True, exist_ok=True)   # make sure the save folder exists
    years = []                                  # one table per year
    for year in range(START_YEAR, END_YEAR + 1):    # range stops before its end, hence the + 1
        path = folder / f"{prefix}_{year}.csv"      # e.g. data/raw/chicago_crash_data/crashes_2018.csv
        if path.exists():                           # saved earlier: read it instead of downloading
            print(f"\nLoading saved {label} for {year}...")         # progress message (\n = blank line first)
            # dtype keeps ID columns as text; low_memory=False reads each column's type consistently.
            data = pd.read_csv(path, dtype=dtypes, low_memory=False)
        else:                                       # not saved yet: download this year
            print(f"\nDownloading {label} for {year}...")           # progress message
            data = socrata_all(dataset_id, {        # the query sent to the City's server:
                "$select": ",".join(columns),       # only the columns we need (smaller, faster download)
                # only crashes dated within this calendar year
                "$where": f"crash_date >= '{year}-01-01T00:00:00' AND crash_date < '{year + 1}-01-01T00:00:00'",
                "$order": ":id",  # the portal's internal row ID: the cheapest, most stable order for paging
            }, page_size=page_size, show_progress=True)     # rows per page; print progress
            data.to_csv(path, index=False)  # saved only after the whole year downloads
            print(f"Saved {len(data):,} records to {path}")         # confirm the save
        years.append(data)                          # keep this year's table
    return pd.concat(years, ignore_index=True)      # one table covering every year


# 1. Fields kept from the crash dataset (each name is a column in the City's Traffic Crashes table).
COLUMNS = [  # the list of column names
    "crash_record_id", "crash_date", "posted_speed_limit", "traffic_control_device", "device_condition",
    "weather_condition", "lighting_condition", "first_crash_type", "trafficway_type", "roadway_surface_cond",
    "road_defect", "intersection_related_i", "hit_and_run_i", "crash_type", "damage", "prim_contributory_cause",
    "num_units", "most_severe_injury", "injuries_total", "injuries_fatal", "injuries_incapacitating",
    "injuries_non_incapacitating", "injuries_reported_not_evident", "latitude", "longitude", "street_name",
    "street_no", "street_direction", "beat_of_occurrence", "location",
]
CRASH_FOLDER = RAW_DIR / "chicago_crash_data"       # where the yearly crash files are saved

# 2. Load or download 2018–2025 and combine the years ("85ca-t3if" is the Traffic Crashes dataset ID).
df = load_years("85ca-t3if", COLUMNS, CRASH_FOLDER, "crashes", "crash data", page_size=10000,
                dtypes={"crash_record_id": "string"})   # read the crash ID as text, not a number

# 3. Standardize types: text IDs, numeric counts and coordinates, and timestamps.
df["crash_record_id"] = df["crash_record_id"].astype("string")      # IDs are labels, not numbers
# Columns that should hold numbers (speed limits, injury counts, coordinates, number of vehicles).
numeric_columns = ["posted_speed_limit", "injuries_total", "injuries_fatal", "injuries_incapacitating",
                   "injuries_non_incapacitating", "injuries_reported_not_evident", "latitude", "longitude", "num_units"]
# Convert text to numbers; errors="coerce" turns anything unreadable into NaN (missing) instead of failing.
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")  # real dates allow hours, months, sorting

# 4. Quick checks that the load worked.
print("\nCrash-data loading complete.")                            # heading for the summary
print(f"Total rows: {len(df):,}")                                   # number of rows loaded
print(f"Unique crash IDs: {df['crash_record_id'].nunique():,}")     # equal to total rows = no repeated IDs
print(f"Date range: {df['crash_date'].min()} to {df['crash_date'].max()}")  # earliest and latest crash
print(f"Local CSV folder: {CRASH_FOLDER.resolve()}")                # where the yearly files are stored


Loading saved crash data for 2018...

Loading saved crash data for 2019...

Loading saved crash data for 2020...

Loading saved crash data for 2021...

Loading saved crash data for 2022...

Loading saved crash data for 2023...

Loading saved crash data for 2024...

Loading saved crash data for 2025...

Crash-data loading complete.
Total rows: 877,919
Unique crash IDs: 877,919
Date range: 2018-01-01 00:00:00 to 2025-12-31 23:52:00
Local CSV folder: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_crash_data


### 2.2 Download or load the People dataset

The People dataset contains person-level injury classifications. These records will be aggregated to one row per crash before being joined to crash-level data.

In [3]:
# Download (or load) the Traffic Crashes – People dataset (one row per person involved), one year at a time.
# The columns to keep: who the person is, which crash and vehicle, their role and their injury level.
PEOPLE_COLUMNS = ["person_id", "crash_record_id", "crash_date", "person_type", "injury_classification", "vehicle_id"]
id_columns = ["person_id", "crash_record_id", "vehicle_id"]         # identifiers to keep as text

# Reuse load_years from the previous cell with the People dataset ID ("u6pd-qa9d"). A smaller page size suits
# this dataset's much larger number of rows. {col: "string" for col in ...} builds {"person_id": "string", ...}.
people = load_years("u6pd-qa9d", PEOPLE_COLUMNS, RAW_DIR / "chicago_people_data", "people", "People records",
                    page_size=5000, dtypes={col: "string" for col in id_columns})   # 5,000 rows per page
people[id_columns] = people[id_columns].astype("string")           # text IDs, so long numbers are not altered

print("\nDownload complete.")                                       # heading for the summary
print("People records:", f"{len(people):,}")                        # number of person rows
print("Unique person IDs:", f"{people['person_id'].nunique():,}")   # equal to rows = no repeated person IDs


Loading saved People records for 2018...

Loading saved People records for 2019...

Loading saved People records for 2020...

Loading saved People records for 2021...

Loading saved People records for 2022...

Loading saved People records for 2023...

Loading saved People records for 2024...

Loading saved People records for 2025...

Download complete.
People records: 1,927,651
Unique person IDs: 1,927,651


### 2.3 Review source categories

Inspect the date coverage and the actual injury classification labels before defining serious outcomes. This helps prevent category-label assumptions from silently affecting the results.

In [4]:
# Check the date coverage and the injury and person-type labels before defining serious outcomes.
print("Crash date range:")                                          # heading
print(df["crash_date"].agg(["min", "max"]))                         # earliest and latest crash
print("\nInjury classifications:")                                  # heading
display(people["injury_classification"].value_counts(dropna=False))  # how often each label appears,
print("\nPerson types:")                                              # including missing (dropna=False)
display(people["person_type"].value_counts(dropna=False))           # e.g. DRIVER, PASSENGER, PEDESTRIAN

Crash date range:
min   2018-01-01 00:00:00
max   2025-12-31 23:52:00
Name: crash_date, dtype: datetime64[us]

Injury classifications:


injury_classification
NO INDICATION OF INJURY     1742917
NONINCAPACITATING INJURY     103093
REPORTED, NOT EVIDENT         62223
INCAPACITATING INJURY         17779
FATAL                          1083
NaN                             556
Name: count, dtype: int64


Person types:


person_type
DRIVER                 1494890
PASSENGER               391030
PEDESTRIAN               24224
BICYCLE                  15520
NON-MOTOR VEHICLE         1687
NON-CONTACT VEHICLE        300
Name: count, dtype: int64

### 2.4 Download or load road-network and traffic-volume data

Crash counts alone cannot show whether a place is dangerous or simply busy. To estimate **vehicle-miles travelled (VMT)**, the notebook combines:

- **Street centerlines:** every street block in Chicago, used as the unit for exposure and for matching crashes to streets and intersections.
- **IDOT AADT:** Illinois DOT's two-way annual average daily traffic on counted road segments (count years vary by segment).
- **City camera ADT:** the City's directional 30-day-average traffic counts from traffic and speed cameras (October 2020 onward), averaged over the available period for each segment.

In [5]:
# Download (or load) the road network and traffic-volume data. Each file is saved after its first download.
EXPOSURE_FOLDER = RAW_DIR / "chicago_exposure_data"     # all exposure, boundary and population files go here
EXPOSURE_FOLDER.mkdir(exist_ok=True)                    # create it if it does not exist yet


# Helper: "download once, then reuse the saved file".
def cached(name, loader, geo=False):
    """Load EXPOSURE_FOLDER/<name> if it was saved before; otherwise run loader() and save the result."""
    path = EXPOSURE_FOLDER / f"{name}.{'geojson' if geo else 'csv'}"  # map data as GeoJSON, tables as CSV
    if path.exists():                                                  # saved earlier: just read it
        print(f"Loading saved {name}: {path}")                        # say which file is being used
        return gpd.read_file(path) if geo else pd.read_csv(path, low_memory=False)   # map file or table
    print(f"Downloading {name}...")                                   # not saved yet: say so
    data = loader()                       # loader is a function (usually a lambda) that does the download
    data.to_file(path, driver="GeoJSON") if geo else data.to_csv(path, index=False)   # save for next time
    print(f"Saved {len(data):,} {'features' if geo else 'rows'} to {path}")           # confirm the save
    return data                                                       # hand the data back to the caller


# Helper: download a map service page by page (like socrata_all, but for GeoJSON map data).
def paged_geojson(url, params, limit_key, offset_key, label, page_size):
    """Page through a GeoJSON web service and return all features as one GeoDataFrame."""
    pages, offset = [], 0                                    # collected pages; features received so far
    while True:                                              # keep going until the last page
        # limit_key/offset_key differ by service ("$limit"/"$offset" for the City, "resultRecordCount"/
        # "resultOffset" for IDOT's ArcGIS service), so they are passed in.
        payload = request_with_retries(url, dict(params, **{limit_key: page_size, offset_key: offset})).json()
        if "error" in payload:                               # ArcGIS reports problems inside the reply
            raise RuntimeError(f"Service error: {payload['error']}")   # stop and show the service's message
        features = payload.get("features", [])               # the map shapes on this page
        if not features:                                     # nothing returned: finished
            break                                            # leave the loop
        # Turn the features into a GeoDataFrame in latitude/longitude (EPSG:4326).
        pages.append(gpd.GeoDataFrame.from_features(features, crs="EPSG:4326"))
        offset += len(features)                              # the next page starts after these features
        print(f"Retrieved {offset:,} {label}")               # progress message
        if len(features) < page_size:                        # a short page is the last page
            break                                            # leave the loop
    # Stack the pages into one GeoDataFrame whose map column is called "geometry".
    return gpd.GeoDataFrame(pd.concat(pages, ignore_index=True), geometry="geometry", crs="EPSG:4326")


# 1. City street centerlines: every street block, with road class, address range and intersection node IDs.
#    fnode_id/tnode_id are the IDs of the intersections at each end of the block.
centerlines = cached("street_centerlines", lambda: paged_geojson(
    f"{SOCRATA_BASE}/pr57-gg9e.geojson",                                    # the centerline dataset's address
    # The fields to download (names, address ranges, road class and the line shape), in a fixed order.
    {"$select": "objectid,fnode_id,tnode_id,pre_dir,street_nam,street_typ,suf_dir,"
                "l_f_add,l_t_add,r_f_add,r_t_add,class,the_geom", "$order": "objectid"},
    "$limit", "$offset", "centerline segments", page_size=50000), geo=True)  # City-style paging; save as map

# 2. IDOT annual average daily traffic (two-way volumes) on road segments inside the Chicago screening box.
#    "geometry" is the box (west, south, east, north) that limits the request to the Chicago area.
idot_aadt = cached("idot_aadt_chicago", lambda: paged_geojson(
    # IDOT's traffic-count map service (ArcGIS REST).
    "https://services2.arcgis.com/aIrBD8yn1TDTEXoz/ArcGIS/rest/services/ADT_COMP_2020Dissolved/FeatureServer/0/query",
    # Cook County segments with a positive count, inside the Chicago box:
    {"where": "COUNTY_NAM='COOK' AND AADT > 0", "geometry": "-87.95,41.64,-87.52,42.03",
     # the box is a rectangle given in latitude/longitude, and segments touching it are included
     "geometryType": "esriGeometryEnvelope", "inSR": 4326, "spatialRel": "esriSpatialRelIntersects",
     # return the ID, volume and count year, plus the line shape, in latitude/longitude
     "outFields": "OBJECTID,AADT,AADT_YR", "outSR": 4326, "returnGeometry": "true",
     "orderByFields": "OBJECTID", "f": "geojson"},                          # fixed order; GeoJSON format
    "resultRecordCount", "resultOffset", "IDOT AADT segments", page_size=1000), geo=True)  # ArcGIS paging
# Volumes and count years arrive as text: convert to numbers.
idot_aadt[["AADT", "AADT_YR"]] = idot_aadt[["AADT", "AADT_YR"]].apply(pd.to_numeric, errors="coerce")

# 3. City camera-based traffic counts (directional, Oct 2020 onward), averaged per segment on the City's server.
#    $select/$group make the server average millions of daily rows into one row per segment and direction.
#    Aliases differ from column names so SoQL does not confuse them with the source columns.
camera_adt = cached("camera_adt_by_segment", lambda: socrata_all("gc7y-n4xa", {
    # segment details, average start/end coordinates, average daily vehicle count, and number of days counted
    "$select": ("segmentid, roadname, direction, fromsegment, tosegment, avg(startlat) as start_lat, "
                "avg(startlon) as start_lon, avg(endlat) as end_lat, avg(endlon) as end_lon, "
                "avg(vehiclecount) as adt, count(*) as days"),
    "$where": f"vehiclecount > 0 AND timestamp < '{END_YEAR + 1}-01-01T00:00:00'",  # real counts, study period
    "$group": "segmentid, roadname, direction, fromsegment, tosegment",    # one row per segment and direction
    "$order": "segmentid",                                                  # fixed order for paging
}).rename(columns={"start_lat": "startlat", "start_lon": "startlon", "end_lat": "endlat", "end_lon": "endlon"}))  # rename the aliases back
camera_cols = ["startlat", "startlon", "endlat", "endlon", "adt", "days"]  # columns that should be numbers
camera_adt[camera_cols] = camera_adt[camera_cols].apply(pd.to_numeric, errors="coerce")  # text -> numbers

# Summary of what was loaded.
print(f"\nStreet centerline segments: {len(centerlines):,}")              # number of street blocks
print(f"IDOT AADT segments in the Chicago screening box: {len(idot_aadt):,} "  # count and year range
      f"(count years {int(idot_aadt['AADT_YR'].min())}–{int(idot_aadt['AADT_YR'].max())})")  # + year range
print(f"Camera ADT directional segments: {len(camera_adt):,} "  # count and typical length
      f"(median {camera_adt['days'].median():.0f} days of counts per segment)")  # + typical count length

Loading saved street_centerlines: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/street_centerlines.geojson
Loading saved idot_aadt_chicago: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/idot_aadt_chicago.geojson
Loading saved camera_adt_by_segment: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/camera_adt_by_segment.csv

Street centerline segments: 56,338
IDOT AADT segments in the Chicago screening box: 4,207 (count years 2016–2025)
Camera ADT directional segments: 2,269 (median 793 days of counts per segment)


### 2.5 Download or load mobility and road-user activity data

Traffic counts give *daily* volumes. To estimate *when* people travel, and to measure walking and cycling activity, the notebook adds:

- **Ride-hail (TNP) trips** aggregated on the City's server by pickup community area and hour. One full week in each season (the first seven days of January, April, July and October) of 2019 and 2024 is used: 56 days and about 14.4 million trips. These trips are used only for the *shape* of travel across the hours of the week; traffic volumes come from the traffic counts. They show how vehicle travel is spread across the week and how that pattern differs between areas.
- **CTA 'L' station entries** (average daily entries per station, 2018–2025) and **bus-stop boardings** (average weekday boardings per stop), used as an indicator of pedestrian activity.
- **Divvy bike-share trips** (2018–2019, the years available on the City portal) by start station and by hour, used as an indicator of cycling activity.

These are *indicators* of activity, not complete counts of every driver, pedestrian or cyclist. Notebook 2 (Section 9) discusses their limitations.

In [6]:
# Download (or load) mobility and road-user activity data used as exposure measures:
#   - ride-hail trips: how vehicle travel is spread across the hours of the week, overall and by area;
#   - CTA 'L' station entries and bus-stop boardings: an indicator of pedestrian activity around transit;
#   - Divvy bike-share trips: an indicator of cycling activity.
# The large ride-hail and bike-share tables are summarised on the City's server in small pieces (one day or one
# month per request), several at once. Each piece is saved in data/raw/chicago_exposure_data/download_chunks, so an
# interrupted run resumes where it stopped.
CHUNK_FOLDER = EXPOSURE_FOLDER / "download_chunks"      # one small CSV per downloaded piece
CHUNK_FOLDER.mkdir(exist_ok=True)                       # create the folder if needed
PARALLEL_REQUESTS = 6                                   # how many pieces to download at the same time
_thread_state = threading.local()                       # storage that is separate for each parallel worker


# Helper: download (or load) one small summary piece and save it.
def cached_chunk(name, dataset_id, params, max_retries=8):
    """Run one small summary query (with its own HTTP session per thread) and save the result."""
    path = CHUNK_FOLDER / f"{name}.csv"                 # where this piece is saved
    if path.exists():                                   # this piece was saved on an earlier run
        return pd.read_csv(path, low_memory=False)      # reuse it
    if not hasattr(_thread_state, "session"):           # first job for this worker: give it a connection
        _thread_state.session = requests.Session()      # a connection used only by this worker
        _thread_state.session.headers.update(HTTP.headers)   # same User-Agent as the main connection
    for attempt in range(max_retries):                  # try up to 8 times
        try:                                            # errors below jump to the except block
            # timeout=(20, 600): summaries can take the server a few minutes to calculate.
            response = _thread_state.session.get(f"{SOCRATA_BASE}/{dataset_id}.json",
                                                 params=dict(params, **{"$limit": 50000}), timeout=(20, 600))  # this piece's query
            if response.status_code == 400:             # a wrong query will not succeed on retry
                raise ValueError(f"Invalid API query for {name}: {response.text[:1000]}")   # stop now
            if response.status_code == 429 or response.status_code >= 500:   # busy server: treat like a
                raise requests.exceptions.ConnectionError(f"HTTP {response.status_code}")  # dropped connection
            response.raise_for_status()                 # stop on any other error code
            part = pd.DataFrame(response.json())        # the summary rows for this piece
            part.to_csv(path, index=False)              # save so it is never downloaded again
            return part                                 # done
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):   # timed out or busy
            if attempt == max_retries - 1:              # no tries left: stop with instructions
                raise RuntimeError(f"{name}: failed after {max_retries} attempts. Rerun the cell to resume.")  # stop with instructions
            time.sleep(min(5 * (2 ** attempt), 120))    # wait longer after each failure (max 2 minutes)


# Helper: download many pieces at the same time.
def fetch_chunks(jobs, label):
    """Download many (name, dataset, query) pieces in parallel, reusing saved pieces; returns {name: DataFrame}."""
    saved = sum((CHUNK_FOLDER / f"{name}.csv").exists() for name, _, _ in jobs)   # True counts as 1
    print(f"{label}: {saved:,} of {len(jobs):,} pieces already saved; downloading {len(jobs) - saved:,} "  # starting message
          f"({PARALLEL_REQUESTS} at a time)...")                                 # starting message
    results, started = {}, time.time()                                           # results; start time
    with ThreadPoolExecutor(max_workers=PARALLEL_REQUESTS) as pool:              # a pool of 6 workers
        futures = {pool.submit(cached_chunk, *job): job[0] for job in jobs}       # start every job
        for done, future in enumerate(as_completed(futures), start=1):           # handle each as it finishes
            results[futures[future]] = future.result()                            # store by piece name
            if done % 10 == 0 or done == len(jobs):                               # progress every 10 pieces
                # pieces finished so far and minutes elapsed
                print(f"  {label}: {done:,}/{len(jobs):,} pieces ready ({(time.time() - started) / 60:.1f} min elapsed)")
    return results                                                               # {piece name: table}


# Helper: split a combined location column into separate latitude and longitude columns.
def add_lat_lon(frame, column="location"):
    """Split a Socrata point column into lat/lon (some City datasets store [lat, lon], others [lon, lat])."""
    # Inner helper: read one location value and return (latitude, longitude).
    def parse(value):
        if isinstance(value, str):                      # loaded from CSV: the point is stored as text
            try:                                        # attempt to read the text
                value = json.loads(value.replace("'", '"'))  # turn the text back into a Python dictionary
            except ValueError:                          # the text could not be read
                return np.nan, np.nan                   # unreadable: treat as missing
        if isinstance(value, dict) and "coordinates" in value:     # GeoJSON-style point
            a, b = (float(v) for v in value["coordinates"][:2])    # the first two numbers
            return (a, b) if 40 < a < 43 else (b, a)    # Chicago's latitude is about 41-42: use it to tell
        if isinstance(value, dict) and "latitude" in value:        # which number is the latitude
            return float(value["latitude"]), float(value["longitude"])   # named latitude/longitude fields
        return np.nan, np.nan                           # anything else: missing
    frame[["lat", "lon"]] = frame[column].apply(parse).tolist()   # two new columns from each (lat, lon) pair
    return frame.drop(columns=column)                   # the original combined column is no longer needed


# 1. Ride-hail trips by pickup community area and hour, one day per request. The sample is one full week in each
#    season (1–7 January, April, July and October) of 2019 (pre-pandemic) and 2024 (most recent full year):
#    56 days, each weekday appearing 8 times. Traffic volumes themselves come from the counts in code cell 5.
TNP_SOURCES = {2019: "m6dm-c72p", 2024: "n26f-ihde"}    # the City publishes each period as its own dataset


# Download (or load) the 56 daily ride-hail summaries and combine them into one table.
def load_tnp_profile():
    # Every (year, dataset, day) to download: 2 years x 4 months x 7 days = 56 days.
    days = [(year, dataset_id, day) for year, dataset_id in TNP_SOURCES.items() for month in [1, 4, 7, 10]
            for day in pd.date_range(f"{year}-{month:02d}-01", periods=7, freq="D")]  # 7 days from the 1st
    # One server-side summary per day: trips and trip-miles by pickup area and hour.
    pieces = fetch_chunks([(f"tnp_{day:%Y-%m-%d}", dataset_id, {
        # pickup area, hour of day, number of trips and total trip-miles
        "$select": ("pickup_community_area, date_extract_hh(trip_start_timestamp) as hour, "
                    "count(*) as trip_count, sum(trip_miles) as trip_miles_sum"),
        # trips starting on this day, with a sensible length (more than 0 and under 100 miles)
        "$where": (f"trip_start_timestamp >= '{day:%Y-%m-%dT%H:%M:%S}' AND trip_start_timestamp < "
                   f"'{day + pd.Timedelta(days=1):%Y-%m-%dT%H:%M:%S}' AND trip_miles > 0 AND trip_miles < 100"),
        "$group": "pickup_community_area, date_extract_hh(trip_start_timestamp)",   # one row per area and hour
    }) for year, dataset_id, day in days], "Ride-hail days")   # one job per day; label for progress messages
    # Stack the 56 daily summaries, labelling each with its year, date and weekday.
    return pd.concat([
        pieces[f"tnp_{day:%Y-%m-%d}"].rename(columns={"trip_count": "trips", "trip_miles_sum": "trip_miles"})  # rename the columns
        .assign(year=year, date=day.date(), day_index=day.dayofweek)  # day_index: Monday = 0
        for year, _, day in days], ignore_index=True)  # for each of the 56 days


tnp = cached("tnp_hourly_profile_seasonal_weeks", load_tnp_profile)  # combined result is saved as one file
tnp_cols = ["pickup_community_area", "hour", "trips", "trip_miles", "day_index", "year"]   # numeric columns
tnp[tnp_cols] = tnp[tnp_cols].apply(pd.to_numeric, errors="coerce")  # text -> numbers
tnp = tnp.dropna(subset=["day_index", "hour"]).astype({"day_index": int, "hour": int})  # whole-number labels

# 2. CTA 'L' stations: locations, and average daily entries per station over 2018–2025.
l_stops = cached("cta_l_stops", lambda: add_lat_lon(socrata_all("8pix-ypme", {   # station platforms
    "$select": "map_id, location", "$order": "map_id"})))                         # station ID and location
l_entries = cached("cta_l_station_entries", lambda: socrata_all("5neh-572f", {   # daily station entries
    "$select": "station_id, avg(rides) as avg_daily_entries",              # averaged on the server
    "$where": f"date >= '{START_YEAR}-01-01T00:00:00' AND date < '{END_YEAR + 1}-01-01T00:00:00'",  # 2018–2025
    "$group": "station_id", "$order": "station_id"}))                      # one row per station
# A station has one row per platform: average them into one point, then attach its ridership.
l_stations = (l_stops.groupby("map_id", as_index=False)[["lat", "lon"]].mean()
              .merge(l_entries.rename(columns={"station_id": "map_id"}), on="map_id"))  # then attach ridership
l_stations["avg_daily_entries"] = pd.to_numeric(l_stations["avg_daily_entries"], errors="coerce")  # numbers

# 3. CTA bus stops: average weekday boardings per stop (the City's stop-level release).
bus_stops = cached("cta_bus_stop_boardings", lambda: add_lat_lon(socrata_all("mq3i-nnqe", {
    "$select": "stop_id, boardings, location", "$order": "stop_id"})))    # stop ID, boardings and location
bus_stops["boardings"] = pd.to_numeric(bus_stops["boardings"], errors="coerce")   # text -> numbers

# 4. Divvy bike-share trips by start station and by hour of day, one month per request. The City portal holds
#    Divvy trips only up to 2019, so 2018–2019 is the only period that overlaps the crash data.
DIVVY_MONTHS = pd.date_range("2018-01-01", "2019-12-01", freq="MS")  # "MS" = the first day of each month


# Helper: one server-side summary per month (24 pieces), stacked into one table of numbers.
def divvy_chunks(prefix, select, group):
    return pd.concat(fetch_chunks([(f"{prefix}_{m:%Y-%m}", "fg6s-gzvg", {   # one job per month
        "$select": select, "$group": group,                                  # what to summarise, and by what
        # trips that started in this month
        "$where": f"start_time >= '{m:%Y-%m-%dT%H:%M:%S}' AND "
                  f"start_time < '{m + pd.offsets.MonthBegin(1):%Y-%m-%dT%H:%M:%S}'"}) for m in DIVVY_MONTHS],
        # label for progress messages; then stack the pieces and convert every column to numbers
        f"Divvy {prefix.split('_')[1]} months").values(), ignore_index=True).apply(pd.to_numeric, errors="coerce")


# Trips per station: add up the monthly counts and average each station's location.
divvy_stations = cached("divvy_trips_by_station", lambda: divvy_chunks(
    # station ID, its average location, and the number of trips starting there
    "divvy_stations", "from_station_id, avg(from_latitude) as station_lat, avg(from_longitude) as station_lon, "
    "count(*) as trip_count", "from_station_id")
    .groupby("from_station_id", as_index=False)                             # combine the 24 months per station
    .agg(lat=("station_lat", "mean"), lon=("station_lon", "mean"), trips=("trip_count", "sum")))  # add up the months
# Trips per hour of day (0-23), added up over all months.
divvy_hourly = cached("divvy_trips_by_hour", lambda: divvy_chunks(
    # hour of day and number of trips starting in that hour
    "divvy_hourly", "date_extract_hh(start_time) as hour, count(*) as trip_count", "date_extract_hh(start_time)")
    .groupby("hour", as_index=False)["trip_count"].sum().rename(columns={"trip_count": "trips"}))  # total/hour

# Summary of what was loaded.
print(f"\nRide-hail sample: {tnp['trips'].sum():,.0f} trips and {tnp['trip_miles'].sum():,.0f} trip-miles "
      f"({', '.join(map(str, TNP_SOURCES))}; {tnp['date'].nunique():,} sampled days)")   # trips, miles, days
print(f"CTA 'L' stations with ridership: {len(l_stations):,}; "  # stations, entries
      f"average daily entries citywide: {l_stations['avg_daily_entries'].sum():,.0f}")   # stations, entries
print(f"CTA bus stops with boardings: {bus_stops['boardings'].notna().sum():,}; "  # stops, boardings
      f"average weekday boardings: {bus_stops['boardings'].sum():,.0f}")                  # stops, boardings
print(f"Divvy stations: {len(divvy_stations):,}; trips 2018–2019: {divvy_stations['trips'].sum():,.0f}")  # bikes

Loading saved tnp_hourly_profile_seasonal_weeks: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/tnp_hourly_profile_seasonal_weeks.csv
Loading saved cta_l_stops: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/cta_l_stops.csv
Loading saved cta_l_station_entries: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/cta_l_station_entries.csv
Loading saved cta_bus_stop_boardings: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/cta_bus_stop_boardings.csv
Loading saved divvy_trips_by_station: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/divvy_trips_by_station.csv
Loading saved divvy_trips_by_hour: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/divvy_trips_by_hour.csv

Ride-hail s

### 2.6 Download or load community-area boundaries and population

Chicago is divided into 77 official **community areas**. Their boundaries are used to assign each crash to an area, and the 2023 population estimates support a per-resident comparison in Notebook 2. Both files are saved after the first download.

In [7]:
# Load the official community-area boundaries and 2023 population estimates (saved after the first download).
# The City's download address for the boundary file (GeoJSON format).
BOUNDARY_URL = "https://data.cityofchicago.org/api/v3/views/igwz-8jzy/query.geojson?accessType=DOWNLOAD"
community_areas = cached("community_area_boundaries", lambda: gpd.read_file(BOUNDARY_URL), geo=True)  # 77 shapes

# Standardize the name and number fields so they match across datasets.
community_areas = community_areas.rename(columns={"community": "community_name", "area_numbe": "community_number"})
# Trim stray spaces and use capitals, so "Austin " and "AUSTIN" count as the same area.
community_areas["community_name"] = community_areas["community_name"].astype("string").str.strip().str.upper()
community_areas["community_number"] = pd.to_numeric(community_areas["community_number"], errors="coerce")  # number
community_areas = community_areas.to_crs("EPSG:4326")  # make sure coordinates are plain latitude/longitude

# Population: keep the 2023 estimate for each area.
population_raw = cached("community_area_population",                  # download once, then reuse
                        lambda: pd.DataFrame(request_with_retries(f"{SOCRATA_BASE}/t68z-cikk.json",  # ask the City for
                                                                  {"$limit": 500}).json()))   # all rows
# Keep 2023 rows and two columns, then give them the names used everywhere else.
population = (population_raw.loc[population_raw["acs_year"].astype(str).str.strip().eq("2023"),
                                 ["community_area", "total_population"]]  # keep these 2 columns
              .rename(columns={"community_area": "community_name", "total_population": "population"}))  # then rename them
population["community_name"] = population["community_name"].astype("string").str.strip().str.upper()  # tidy
population["population"] = pd.to_numeric(population["population"], errors="coerce")   # text -> numbers
population = population.loc[population["population"].gt(0)]     # drop blank or zero estimates

print(f"Community areas in boundary file: {len(community_areas)}")           # should be 77
print(f"Community areas with a 2023 population estimate: {len(population)}")  # should be 77

Loading saved community_area_boundaries: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/community_area_boundaries.geojson
Loading saved community_area_population: /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/raw/chicago_exposure_data/community_area_population.csv
Community areas in boundary file: 77
Community areas with a 2023 population estimate: 77


## 3. Data quality and preparation

### 3.1 Normalize fields and audit crash IDs

The original downloaded crash data is preserved in `df`. Repeated IDs are audited before reducing the data to one row per crash for analysis.

In [8]:
# Audit repeated crash IDs: count them and check whether any repeated ID has conflicting key fields.
duplicate_mask = df["crash_record_id"].duplicated(keep=False)   # True for EVERY row whose ID appears twice+
# Fields that should be identical if a repeated ID really is the same crash.
audit_fields = ["crash_date", "latitude", "longitude", "injuries_total", "injuries_fatal",
                "injuries_incapacitating", "most_severe_injury"]
# For each repeated ID, count distinct values in each key field; more than 1 (gt(1)) in any field
# (.any(axis=1) checks across the columns of a row) means the copies disagree. .sum() counts those IDs.
conflicting_ids = (df.loc[duplicate_mask].groupby("crash_record_id")[audit_fields]
                   .nunique(dropna=False).gt(1).any(axis=1).sum())  # count conflicting IDs

print(f"Downloaded rows: {len(df):,}")                                          # all rows
print(f"Rows flagged for repeated crash IDs: {int(duplicate_mask.sum()):,}")    # rows sharing an ID
print(f"Repeated IDs with conflicting key fields: {int(conflicting_ids):,}")    # IDs whose copies disagree

Downloaded rows: 877,919
Rows flagged for repeated crash IDs: 0
Repeated IDs with conflicting key fields: 0


### 3.2 Document cleaning decisions

Missing dates prevent time-based analysis. Missing or invalid coordinates prevent spatial analysis but do not automatically exclude a crash from non-geographic analysis. The broad geographic bounds below are a screening rule, not an official city boundary.

In [9]:
# Summarise data-quality issues and the cleaning decision taken for each.
# A broad box around Chicago: points outside it are almost certainly mistakes.
in_bounds = df["latitude"].between(41.60, 42.10) & df["longitude"].between(-87.95, -87.50)
has_coordinates = df["latitude"].notna() & df["longitude"].notna()  # & = "and"; notna() = not missing

# Each tuple is one row of the table: (issue, number of rows affected, decision). ~ means "not".
audit = pd.DataFrame([
    ("Repeated crash IDs", duplicate_mask.sum(),                                    # rows sharing an ID
     "Inspect key-field conflicts; count each crash ID once in summaries."),
    ("Missing or invalid crash date", df["crash_date"].isna().sum(), "Exclude from time-based analyses."),  # rows with no date
    ("Missing latitude or longitude", (~has_coordinates).sum(),                     # rows without a location
     "Retain for non-geographic analyses; exclude from spatial joins."),
    ("Outside broad Chicago screening bounds", (has_coordinates & ~in_bounds).sum(),  # located, but outside box
     "Review; exclude from spatial analysis unless verified."),
    ("Zero latitude or longitude", (df["latitude"].eq(0) | df["longitude"].eq(0)).sum(),   # | = "or"
     "Treat as invalid for geographic analysis."),
], columns=["issue", "rows_flagged", "decision"]).astype({"rows_flagged": int})    # column names; whole numbers
display(audit)                                                                      # show the table

,issue,rows_flagged,decision
0,Repeated crash IDs,0,Inspect key-field conflicts; count each crash ...
1,Missing or invalid crash date,0,Exclude from time-based analyses.
2,Missing latitude or longitude,7075,Retain for non-geographic analyses; exclude fr...
3,Outside broad Chicago screening bounds,55,Review; exclude from spatial analysis unless v...
4,Zero latitude or longitude,55,Treat as invalid for geographic analysis.


### 3.3 Create analysis-ready crash and geographic datasets

The temporal dataset keeps records with valid dates. For crash-level counting, one row per crash ID is used after the duplicate audit.

In [10]:
# Build the analysis datasets: one row per crash with a valid date, and the subset with valid coordinates.
df_clean = (df.loc[df["crash_date"].notna()]                         # keep rows with a usable date
            .sort_values(["crash_record_id", "crash_date"])          # predictable order before de-duplicating
            .drop_duplicates(subset=["crash_record_id"], keep="first"))  # one row per crash ID
# Valid location = inside the Chicago box and not the placeholder value 0.
valid_coordinates = (df_clean["latitude"].between(41.60, 42.10) & df_clean["longitude"].between(-87.95, -87.50)
                     & df_clean["latitude"].ne(0) & df_clean["longitude"].ne(0))   # ne = "not equal to"
df_geo = df_clean.loc[valid_coordinates].copy()   # .copy() = an independent table, safe to change later

print(f"Raw rows: {len(df):,}")                                                  # before cleaning
print(f"Crash-level records with valid dates: {len(df_clean):,}")                # after cleaning
print(f"Records eligible for geographic analysis: {len(df_geo):,}")              # with usable locations
print(f"Date range used: {df_clean['crash_date'].min()} to {df_clean['crash_date'].max()}")   # first/last

Raw rows: 877,919
Crash-level records with valid dates: 877,919
Records eligible for geographic analysis: 870,789
Date range used: 2018-01-01 00:00:00 to 2025-12-31 23:52:00


### 3.4 Prepare person-level injury information

Exact duplicate People rows are removed. Person IDs associated with multiple crash IDs or conflicting injury classifications are flagged for review rather than silently resolved by keeping the first record.

For this project, a **serious-outcome crash** is a crash involving at least one person classified as `FATAL` or `INCAPACITATING INJURY`. A crash outcome is considered known if at least one person has a non-missing classification other than `UNKNOWN`.

In [11]:
# Standardize People labels, remove exact duplicate rows, and flag person IDs with inconsistent records.
for col in ["person_type", "injury_classification"]:                     # the two label columns
    people[col] = people[col].astype("string").str.strip().str.upper()   # consistent spelling of labels

people_rows_before = len(people)                       # remember the count to report how many were removed
people = people.drop_duplicates().reset_index(drop=True)  # drop rows identical in every column; renumber

# For each person ID, count how many different crashes, injury labels and person types it has.
person_review = people.dropna(subset=["person_id"]).groupby("person_id").agg(
    distinct_crash_ids=("crash_record_id", "nunique"),             # different crashes for this person
    distinct_injury_classes=("injury_classification", "nunique"),  # different injury labels
    distinct_person_types=("person_type", "nunique"),              # different roles (driver, passenger...)
    rows=("person_id", "size"),                        # number of rows for this person
)
# A person ID with more than one of anything is flagged for review instead of being silently "fixed".
flagged_people = (person_review[["distinct_crash_ids", "distinct_injury_classes", "distinct_person_types", "rows"]]
                  .gt(1).any(axis=1).sum())            # more than 1 in any column -> flagged; count them

print(f"People rows before removing exact duplicates: {people_rows_before:,}")       # before
print(f"Exact duplicate People rows removed: {people_rows_before - len(people):,}")  # removed
print(f"People rows after removing exact duplicates: {len(people):,}")               # after
print(f"Person IDs flagged for review: {flagged_people:,}")                          # inconsistent IDs

People rows before removing exact duplicates: 1,927,651
Exact duplicate People rows removed: 0
People rows after removing exact duplicates: 1,927,651
Person IDs flagged for review: 0


### 3.5 Aggregate People records to crash level

The People dataset is aggregated before joining it to the crash table, so a crash involving multiple people is still counted once. Unknown classifications are not automatically treated as non-serious. The aggregation also records how many drivers, pedestrians and cyclists each crash involved. These flags are used for the pedestrian and cyclist exposure rates and to convert crash rates into per-driver probabilities.

In [12]:
# Summarise the People data to one row per crash and attach it to the crash table.
# A crash has a serious outcome if anyone was killed or incapacitated; its outcome is "known" if at least
# one person has a classification other than missing or UNKNOWN.
injury_class = people["injury_classification"]                                    # shorter name for reuse
people["classification_known"] = injury_class.notna() & ~injury_class.isin(["UNKNOWN", ""])  # real label
people["is_serious_person"] = injury_class.isin(["FATAL", "INCAPACITATING INJURY"])           # killed/badly hurt
# One True/False column for each road-user type: is_driver, is_pedestrian, is_cyclist.
for flag, person_type in [("is_driver", "DRIVER"), ("is_pedestrian", "PEDESTRIAN"), ("is_cyclist", "BICYCLE")]:
    people[flag] = people["person_type"].eq(person_type)      # True where the person has this role

# One row per crash: "any" = at least one person in the crash; "sum" counts True values (number of drivers).
people_by_crash = people.dropna(subset=["crash_record_id"]).groupby("crash_record_id").agg(
    has_known_outcome=("classification_known", "any"),        # anyone with a real injury label?
    has_serious_outcome=("is_serious_person", "any"),         # anyone killed or incapacitated?
    drivers=("is_driver", "sum"),                             # how many drivers
    pedestrian_involved=("is_pedestrian", "any"),             # any pedestrian?
    cyclist_involved=("is_cyclist", "any"),                   # any cyclist?
).reset_index()                                               # turn crash_record_id back into a column

# Left join: keep every crash, adding the people summary where it exists. validate="one_to_one" stops with an
# error if any crash would appear twice, which protects against double counting.
crash_enriched = df_clean.merge(people_by_crash, on="crash_record_id", how="left", validate="one_to_one")
# serious_outcome is True/False when the outcome is known, and missing (NA) when it is not.
known = crash_enriched["has_known_outcome"].fillna(False).astype(bool)     # crashes with a known outcome
crash_enriched["serious_outcome"] = (crash_enriched["has_serious_outcome"].fillna(False).astype(bool)  # known True/False, else NA
                                     .astype("boolean").where(known))   # "boolean" type allows NA; .where
                                                                        # blanks the rows where known is False
# Crashes with no People rows get False (no pedestrian/cyclist recorded) instead of missing.
crash_enriched[["pedestrian_involved", "cyclist_involved"]] = (
    crash_enriched[["pedestrian_involved", "cyclist_involved"]].fillna(False).astype(bool))  # no record = no

print(f"Crashes with known outcome status: {crash_enriched['serious_outcome'].notna().sum():,}")   # known
print(f"Crashes involving serious outcomes: {crash_enriched['serious_outcome'].eq(True).sum():,}")  # serious
print(f"Crashes involving a pedestrian: {crash_enriched['pedestrian_involved'].sum():,}")           # pedestrian
print(f"Crashes involving a cyclist: {crash_enriched['cyclist_involved'].sum():,}")                 # cyclist
print(f"Average drivers recorded per crash: {crash_enriched['drivers'].mean():.2f}")                # about 1.7

Crashes with known outcome status: 875,925
Crashes involving serious outcomes: 16,023
Crashes involving a pedestrian: 23,040
Crashes involving a cyclist: 15,360
Average drivers recorded per crash: 1.71


### 3.6 Build the road-segment exposure model

Every street block covered by the crash dataset receives a daily traffic volume, in this order of preference:

1. **City camera count:** directional counts on the same street within 15 m, added together for a two-way volume.
2. **IDOT AADT:** the nearest IDOT count segment within 15 m.
3. **Estimate:** the median measured volume for the block's road class (arterial, collector, local street, and so on).

**Vehicle-miles travelled** on each block = daily volume × block length (miles) × number of days in 2018–2025.

Interstate expressways and ramps are excluded because Chicago Police crash reports exclude them. Lake Shore Drive is kept. The coverage table shows how much of each road class has a measured volume. Rates for areas or corridors that depend mostly on estimated volumes are flagged as lower-confidence in Notebook 2.

In [13]:
# Road-segment exposure model: a daily traffic volume and vehicle-miles travelled (VMT) for every street block.
PROJECTED_CRS = "EPSG:26971"  # NAD83 / Illinois East (metres), used for distances and lengths
METERS_PER_MILE = 1609.344    # for converting metres to miles
STUDY_DAYS = (pd.Timestamp(f"{END_YEAR + 1}-01-01") - pd.Timestamp(f"{START_YEAR}-01-01")).days  # 2,922 days
VOLUME_MATCH_DISTANCE_M = 15  # a block takes a counted volume only if the count line is this close

# 1. Standardize street names, road classes, node IDs and address ranges.
streets = centerlines.copy()                                          # work on a copy of the raw centerlines
text_cols = ["pre_dir", "street_nam", "street_typ", "suf_dir", "class"]   # direction, name, type, class
# For each text column: make it text, fill blanks with "", trim spaces, capitalise.
streets[text_cols] = streets[text_cols].apply(lambda s: s.astype("string").fillna("").str.strip().str.upper())
number_cols = ["fnode_id", "tnode_id", "l_f_add", "l_t_add", "r_f_add", "r_t_add"]   # node IDs and addresses
streets[number_cols] = streets[number_cols].apply(pd.to_numeric, errors="coerce")    # text -> numbers
# Full name such as "W 87TH ST": join the parts, then split/join to remove extra spaces from empty parts.
streets["street_full"] = (streets["pre_dir"] + " " + streets["street_nam"] + " " + streets["street_typ"] + " "
                          + streets["suf_dir"]).str.split().str.join(" ")  # join the parts

# 2. Keep the streets covered by Chicago Police crash reports: drop interstates and ramps (outside CPD's
#    jurisdiction), airport/private ways, rivers, unclassified lines and empty geometries. Lake Shore Drive stays.
#    Interstates = class 1 roads that are expressways/ramps or are named like "SKYWAY" or "I 90".
is_interstate = streets["class"].eq("1") & (streets["street_typ"].isin(["EXPY", "RL"])
                                            | streets["street_nam"].str.contains(r"SKYWAY|\bI\d", regex=True))  # expressway, ramp or named interstate
has_geometry = streets.geometry.apply(lambda g: g is not None and not g.is_empty)   # a real line exists
road = streets.loc[~is_interstate & ~streets["class"].isin(["9", "99", "RIV", ""]) & has_geometry]  # keep these
road = road.to_crs(PROJECTED_CRS).reset_index(drop=True)    # switch to metres so lengths are correct
road["segment_id"] = np.arange(len(road))                   # simple ID 0, 1, 2, ... for each block
road["length_m"] = road.geometry.length                     # block length in metres
road = road.loc[road["length_m"] > 1].copy()                # drop tiny slivers (1 m or less)
# Readable road-class names; any other class code becomes "Other street".
road["road_class"] = road["class"].map({"1": "Lake Shore Drive", "2": "Arterial", "3": "Collector",
                                        "4": "Local street", "7": "Lower-level street"}).fillna("Other street")
# The middle point of each block (halfway along the line), used to match counts and areas to blocks.
midpoints = gpd.GeoDataFrame(road[["segment_id", "street_nam"]],
                             geometry=road.geometry.interpolate(0.5, normalized=True), crs=PROJECTED_CRS)  # middle of each line

# 3. City camera counts: match each block to camera links on the same street within 15 m, average each direction,
#    then add the directions together for a two-way volume.
camera_links = camera_adt.dropna(subset=["startlat", "startlon", "endlat", "endlon", "adt"])  # complete rows only
camera_links = gpd.GeoDataFrame(                          # turn the camera table into map lines:
    camera_links[["roadname", "direction", "adt"]],      # keep street name, direction and volume
    # Build a straight line from each counted segment's start point to its end point.
    geometry=[LineString([(x1, y1), (x2, y2)]) for x1, y1, x2, y2 in
              camera_links[["startlon", "startlat", "endlon", "endlat"]].to_numpy()],  # start and end of each segment
    crs="EPSG:4326").to_crs(PROJECTED_CRS)               # from latitude/longitude to metres
camera_links["roadname"] = camera_links["roadname"].astype("string").str.strip().str.upper()   # tidy names
camera_links["geometry"] = camera_links.geometry.buffer(VOLUME_MATCH_DISTANCE_M)  # widen each line by 15 m
# Spatial join: which block midpoints fall inside a widened camera line?
camera_match = gpd.sjoin(midpoints, camera_links, how="inner", predicate="within")
camera_match = camera_match.loc[camera_match["street_nam"].eq(camera_match["roadname"])]  # same street only
# Average each direction separately, then add the directions (e.g. northbound + southbound).
camera_volume = (camera_match.groupby(["segment_id", "direction"])["adt"].mean()
                 .groupby(level="segment_id").sum().rename("camera_adt"))  # add the directions

# 4. IDOT counts: take the nearest IDOT segment within 15 m (most recent count year if tied).
idot_match = gpd.sjoin_nearest(midpoints, idot_aadt[["AADT", "AADT_YR", "geometry"]].to_crs(PROJECTED_CRS),
                               how="inner", max_distance=VOLUME_MATCH_DISTANCE_M, distance_col="idot_distance_m")  # within 15 m
# Sort so the closest (then newest) count comes first for each block, and keep only that one.
idot_volume = (idot_match.sort_values(["segment_id", "idot_distance_m", "AADT_YR"], ascending=[True, True, False])
               .drop_duplicates("segment_id").set_index("segment_id")["AADT"].rename("idot_aadt"))  # keep the best match

# 5. Volume for every block: camera count first, then IDOT count, otherwise the median measured volume for the
#    block's road class. A typical urban volume is used only if a class has fewer than 20 measured blocks.
# Backup volumes (vehicles per day).
FALLBACK_ADT = {"Lake Shore Drive": 60000, "Arterial": 15000, "Collector": 6000, "Local street": 1000,
                "Lower-level street": 5000, "Other street": 1000}   # vehicles per day, used only as a backup
road = road.join(camera_volume, on="segment_id").join(idot_volume, on="segment_id")   # attach both sources
road["adt"] = road["camera_adt"].combine_first(road["idot_aadt"])   # camera value, else the IDOT value
road["volume_measured"] = road["adt"].notna()                       # True where a real count exists
# Median measured volume and number of measured blocks for each road class.
class_stats = road.loc[road["volume_measured"]].groupby("road_class")["adt"].agg(["median", "size"])
# The estimate for each class: its median if at least 20 blocks were measured, otherwise the backup value.
imputed_adt = {cls: class_stats.loc[cls, "median"] if class_stats["size"].get(cls, 0) >= 20 else fallback
               for cls, fallback in FALLBACK_ADT.items()}  # for every road class
road["adt"] = road["adt"].fillna(road["road_class"].map(imputed_adt))   # fill gaps with the class estimate

# 6. Vehicle-miles travelled per day and over the whole 2018–2025 study period.
road["length_mi"] = road["length_m"] / METERS_PER_MILE              # block length in miles
road["daily_vmt"] = road["adt"] * road["length_mi"]                 # vehicles per day x miles
road["study_vmt"] = road["daily_vmt"] * STUDY_DAYS                  # x number of days

# 7. Coverage of measured volumes by road class.
coverage = (road.assign(measured_miles=road["length_mi"].where(road["volume_measured"], 0))  # 0 if estimated
            # per road class: total miles, measured miles, daily VMT and typical volume
            .groupby("road_class").agg(miles=("length_mi", "sum"), measured_miles=("measured_miles", "sum"),
                                       daily_vmt=("daily_vmt", "sum"), median_adt=("adt", "median")))  # totals per class
coverage["pct_miles_measured"] = coverage["measured_miles"] / coverage["miles"] * 100   # % of miles counted
coverage["pct_of_city_vmt"] = coverage["daily_vmt"] / coverage["daily_vmt"].sum() * 100  # share of all driving
coverage = coverage.sort_values("daily_vmt", ascending=False)       # biggest share of driving first

# Share of all estimated VMT that comes from blocks with a real count.
measured_vmt_share = road.loc[road["volume_measured"], "daily_vmt"].sum() / road["daily_vmt"].sum() * 100
print(f"Street blocks in the exposure model: {len(road):,} ({road['length_mi'].sum():,.0f} miles)")  # size
print(f"Blocks with a measured volume: {road['volume_measured'].sum():,} "  # measured share
      f"({measured_vmt_share:.1f}% of estimated VMT comes from measured volumes)")       # data quality
print(f"Estimated daily VMT on CPD-covered streets: {road['daily_vmt'].sum() / 1e6:,.1f} million")   # per day
print(f"Estimated VMT over {START_YEAR}–{END_YEAR}: {road['study_vmt'].sum() / 1e9:,.1f} billion")  # in total
print("Daily volumes assigned to unmeasured blocks:", {k: int(v) for k, v in imputed_adt.items()})  # estimates
display(coverage.round(1))                                          # the coverage table, 1 decimal place

Street blocks in the exposure model: 52,767 (4,116 miles)
Blocks with a measured volume: 18,685 (59.0% of estimated VMT comes from measured volumes)
Estimated daily VMT on CPD-covered streets: 27.2 million
Estimated VMT over 2018–2025: 79.4 billion
Daily volumes assigned to unmeasured blocks: {'Lake Shore Drive': 94500, 'Arterial': 15800, 'Collector': 10908, 'Local street': 3300, 'Lower-level street': 14300, 'Other street': 3200}


,miles,measured_miles,daily_vmt,median_adt,pct_miles_measured,pct_of_city_vmt
road_class,,,,,,
Local street,3089.4,361.0,10730785.4,3300.0,11.7,39.5
Arterial,424.3,407.0,7385632.8,15800.0,95.9,27.2
Collector,473.7,438.5,5548286.7,10908.4,92.6,20.4
Lake Shore Drive,34.3,21.7,3057008.0,94500.0,63.4,11.3
Other street,85.5,3.9,296760.2,3200.0,4.5,1.1
Lower-level street,9.1,8.0,142945.2,14300.0,87.7,0.5


### 3.7 Save the prepared data for Notebook 2

All datasets that Notebook 2 needs are saved together in one file, `data/processed/prepared_data.pkl`. A pickle file keeps column types (dates, text IDs) and map geometry exactly as they are here, so Notebook 2 can start straight from the analysis without repeating the downloads and cleaning.

In [14]:
# Save every dataset that Notebook 2 needs in one file.
prepared = {                               # a dictionary: label -> dataset
    "crash_enriched": crash_enriched,      # one row per crash, with person-level outcome flags
    "df_geo": df_geo[["crash_record_id", "crash_date", "latitude", "longitude", "injuries_total",
                      "injuries_fatal", "first_crash_type"]],  # crashes with valid coordinates
    "road": road,                          # street blocks with traffic volume and vehicle-miles travelled
    "midpoints": midpoints,                # the midpoint of each street block
    "idot_aadt": idot_aadt,                # IDOT traffic counts (used to find much busier roads nearby)
    "tnp": tnp,                            # ride-hail trips by area, weekday and hour
    "l_stations": l_stations, "bus_stops": bus_stops,                 # transit activity
    "divvy_stations": divvy_stations, "divvy_hourly": divvy_hourly,   # bike-share activity
    "community_areas": community_areas, "population": population,     # boundaries and residents
}
PREPARED_FILE = PROCESSED_DIR / "prepared_data.pkl"   # data/processed/prepared_data.pkl
# A pickle file stores Python objects exactly as they are (dates, text IDs, map shapes), unlike a CSV.
pd.to_pickle(prepared, PREPARED_FILE)
print(f"Saved {len(prepared)} datasets to {PREPARED_FILE} ({PREPARED_FILE.stat().st_size / 1e6:,.0f} MB)")  # size
print("Next step: open notebooks/02_analysis.ipynb and run all cells.")                     # what to do next

Saved 12 datasets to /Users/armon/Documents/GitHub Projects/chicago-traffic-crash-analysis/data/processed/prepared_data.pkl (761 MB)
Next step: open notebooks/02_analysis.ipynb and run all cells.
